# `parse_word` — extraction d'un .docx en DataFrames

Module testé : [`src/docpipeline/parsing/word/parse_word.py`](../src/docpipeline/parsing/word/parse_word.py).

## Sortie de `parse_word(docx)`

| Sortie | Contenu |
|---|---|
| `paragraph_df` | 1 ligne = 1 paragraphe (style, alignement, indents, list_level, heading_level, ...) |
| `span_df`      | 1 ligne = 1 run (span_id, text, font_name, font_size_pt, bold, italic, underline, color, highlight, ...) |
| `image_df`     | 1 ligne = 1 image embarquée |
| `table_df`     | 1 ligne = 1 table (n_rows, n_cols, style, alignment) |
| `doc_summary`  | JSON synthèse : metadata, source_tool, has_toc, has_track_changes, has_comments, has_footnotes, ... |

Le `span_id` (format `w_<para_idx>_<run_idx>`) est stable et reproductible — utilisable pour un cycle ultérieur extract → modify → rebuild.

Démo ci-dessous : extraction sur `tests/fixtures/contrat_assurance.docx`.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.word.parse_word import parse_word

DOCX = Path('../tests/fixtures/contrat_assurance.docx')

pd.set_option('display.max_colwidth', 120)

result = parse_word(DOCX)

display(Markdown('### 1. `doc_summary` — synthèse au niveau document'))
summary_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in result['doc_summary'].items()])
display(summary_df)

display(Markdown(f"### 2. `span_df` — {len(result['span_df'])} spans avec leurs styles"))
display(result['span_df'][['span_id', 'text', 'font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color']])

display(Markdown(f"### 3. `paragraph_df` — {len(result['paragraph_df'])} paragraphes"))
display(result['paragraph_df'][['paragraph_index', 'text', 'style_name', 'heading_level', 'alignment', 'is_list_item']])

display(Markdown(f"### 4. `table_df` — {len(result['table_df'])} table(s) native(s)"))
display(result['table_df'])

### 1. `doc_summary` — synthèse au niveau document

,key,value
0,doc_hash,0134552717eb2b56e4ae7b799c060f7e70ed838968e293f3c9ea6c5faa9eb5c2
1,file_size_bytes,37367
2,n_paragraphs,10
3,n_spans,11
4,n_images,0
5,n_tables,1
6,n_sections,1
7,n_headings,6
8,n_list_items,0
9,total_char_count,689


### 2. `span_df` — 11 spans avec leurs styles

,span_id,text,font_name,font_size_pt,bold,italic,underline,color
0,w_0_0,Contrat d'assurance — Conditions Générales,,None,False,False,None,NaN
1,w_1_0,1. Objet du contrat,,None,False,False,None,NaN
2,w_2_0,Le présent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activité professionnelle de l'a...,,None,False,False,None,NaN
3,w_3_0,2. Garanties,,None,False,False,None,NaN
4,w_4_0,2.1 Garantie principale,,None,False,False,None,NaN
5,w_5_0,La franchise applicable est fixée à 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par année...,,None,False,False,None,NaN
6,w_6_0,IMPORTANT :,,None,True,False,None,#FF0000
7,w_6_1,Toute déclaration de sinistre doit intervenir dans un délai de 5 jours ouvrés.,,None,False,False,None,NaN
8,w_7_0,3. Tableau des garanties,,None,False,False,None,NaN
9,w_8_0,4. Exclusions,,None,False,False,None,NaN


### 3. `paragraph_df` — 10 paragraphes

,paragraph_index,text,style_name,heading_level,alignment,is_list_item
0,0,Contrat d'assurance — Conditions Générales,Title,0.0,None,False
1,1,1. Objet du contrat,Heading 1,1.0,None,False
2,2,Le présent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activité professionnelle de l'a...,Normal,NaN,None,False
3,3,2. Garanties,Heading 1,1.0,None,False
4,4,2.1 Garantie principale,Heading 2,2.0,None,False
5,5,La franchise applicable est fixée à 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par année...,Normal,NaN,None,False
6,6,IMPORTANT : Toute déclaration de sinistre doit intervenir dans un délai de 5 jours ouvrés.,Normal,NaN,None,False
7,7,3. Tableau des garanties,Heading 1,1.0,None,False
8,8,4. Exclusions,Heading 1,1.0,None,False
9,9,"Sont exclus du présent contrat les sinistres résultant de : faute intentionnelle, guerre, actes terroristes, catastr...",Normal,NaN,None,False


### 4. `table_df` — 1 table(s) native(s)

,doc_hash,table_index,n_rows,n_cols,style_name,alignment,autofit,n_cells_with_text
0,0134552717eb2b56e4ae7b799c060f7e70ed838968e293f3c9ea6c5faa9eb5c2,0,5,3,Table Grid,None,True,15
